# Prompt activation PCA

Load the completed years activation dataset, select a subset, and compute the first 15 PCs
for every layer in parallel. `N_WORKERS` controls parallelism. The Plotly 3D and 2D
plots have a layer dropdown and prompt hover text.

PCA centers features without scaling or whitening. Each layer has its own basis.
Run the cells in order; edit `ACTIVATION_DIR` to use another extraction.

Set `PCS_3D` and `PCS_2D` at the top of their plotting cells (1-based PC numbers).
For example, `(4, 5, 6)` plots PCs 4–6. Change these and rerun only the plotting cell.

The final **K3D explorer** cell offers opaque 3D points with live layer, PC, and
point-size controls. After computing PCA, you can skip directly to that cell.

In [7]:
from pathlib import Path
from collections import defaultdict
from html import escape
import json
import os
import sys
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from joblib import Parallel, delayed, parallel_config
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from dalg.data.prompt_activations import load_prompt_activation_index

pio.renderers.default = "plotly_mimetype"
ACTIVATION_DIR = REPO_ROOT / "dalg-cache/sae_manifold_temperature_02d_llama31_8b_last"
config = json.loads((ACTIVATION_DIR / "config.json").read_text())
LAYERS = sorted(config["layers"])
# This reader checks completion and joins source records to activation coordinates.
index = load_prompt_activation_index(ACTIVATION_DIR, layer=LAYERS[0])
metadata = pd.DataFrame(index)
SUBSETS = sorted(metadata["subset"].unique())
summary = metadata.groupby("subset").size().rename("prompts").to_frame()
summary["label_fields"] = [
    sorted({key for labels in metadata.loc[metadata.subset.eq(name), "labels"]
            for key in labels})
    for name in summary.index
]
print(f"{config['model']}: {len(metadata):,} prompts, {len(LAYERS)} layers, "
      f"{config['d_model']} activation features; last token")
display(summary)

meta-llama/Llama-3.1-8B: 15,000 prompts, 32 layers, 4096 activation features; last token


,prompts,label_fields
subset,,
temperature,15000,"[celsius, fahrenheit]"


## Select a subset

Edit `SUBSET` to any name in the table above and rerun this cell and everything below.
`COLOR_LABEL = None` chooses a useful ground-truth label automatically; set it to
another field from the table to change the coloring. For `colors`, `"hex"` uses the
actual saved colors. Changing only the coloring requires rerunning this cell and
the plotting cells, without recomputing PCA.

In [10]:
SUBSET = "temperature"  # colors, age, temperature, geography, days, years, formality, sent_length
COLOR_LABEL = None
SEED = 0
N_COMPONENTS = 15
N_WORKERS = min(8, len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity")
                else (os.cpu_count() or 1))  # Increase to run more layers concurrently.

if SUBSET not in SUBSETS:
    raise ValueError(f"Choose SUBSET from {SUBSETS}")
selected = metadata.loc[metadata.subset.eq(SUBSET)].reset_index(drop=True)
labels = pd.json_normalize(selected["labels"].tolist())
DEFAULT_COLOR = {
    "colors": "hue", "age": "age", "temperature": "fahrenheit",
    "geography": "continent", "days": "day_idx", "years": "year",
    "formality": "formality", "sent_length": "n_tokens",
}
color_label = COLOR_LABEL or DEFAULT_COLOR.get(SUBSET, labels.columns[0])
if color_label not in labels:
    raise ValueError(f"Choose COLOR_LABEL from {labels.columns.tolist()}")
print(f"Selected {SUBSET}: {len(selected):,} prompts; color = {color_label}")
display(pd.concat([selected[["source_row_id", "prompt", "decoded_token"]], labels], axis=1).head())

Selected temperature: 15,000 prompts; color = fahrenheit


,source_row_id,prompt,decoded_token,fahrenheit,celsius
0,0,Today it's -30 degrees Fahrenheit outside,outside,-30.00,-34.444444
1,1,Today it's -30.01 degrees Fahrenheit outside,outside,-30.01,-34.450000
2,2,Today it's -30.02 degrees Fahrenheit outside,outside,-30.02,-34.455556
3,3,Today it's -30.03 degrees Fahrenheit outside,outside,-30.03,-34.461111
4,4,Today it's -30.04 degrees Fahrenheit outside,outside,-30.04,-34.466667


## Fit 15 PCs for every layer in parallel

`N_WORKERS` controls how many layers run concurrently (up to eight by default).
Each worker loads only the selected subset, uses one CPU thread, and fits all its
rows, including distinct records with duplicate prompt text. A seeded randomized
SVD estimates the top 15 PCs with 20 oversamples and 7 power iterations.

Completed layers report loading and PCA times immediately, in completion order.
The final tables and layer selector remain ordered by layer number. Each worker
holds one activation matrix, so memory use increases with `N_WORKERS`.
`results` retains scores, PCA models, and matching source row IDs. No extraction
artifacts are modified.

In [11]:
def load_subset_layer(rows, layer):
    """Read selected rows in their metadata order, grouping reads by saved shard."""
    groups = defaultdict(list)
    for output_row, row in enumerate(rows.itertuples(index=False)):
        groups[int(row.shard)].append((output_row, int(row.row_in_shard), int(row.token_slot)))
    values = np.empty((len(rows), config["d_model"]), dtype=np.float32)
    for shard_id, coordinates in groups.items():
        path = ACTIVATION_DIR / f"layer{layer:02d}" / f"shard_{shard_id:05d}.pt"
        shard = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
        if shard.ndim != 3 or tuple(shard.shape[1:]) != (1, config["d_model"]):
            raise ValueError(f"Unexpected activation shape in {path}: {tuple(shard.shape)}")
        output_rows, shard_rows, token_slots = map(list, zip(*coordinates))
        values[output_rows] = shard[shard_rows, token_slots].float().numpy()
    if not np.isfinite(values).all():
        raise ValueError(f"Nonfinite activations in layer {layer}")
    return values



def fit_layer(layer, rows, subset, seed):
    """Load and project one layer; joblib limits each worker to one CPU thread."""
    started = perf_counter()
    x = load_subset_layer(rows, layer)
    loaded = perf_counter()
    if not np.any(x != x[0]):
        raise ValueError(f"Layer {layer} has zero variance across the selected prompts")
    pca = PCA(n_components=N_COMPONENTS, svd_solver="randomized", random_state=seed,
              n_oversamples=20, iterated_power=7)
    scores = pca.fit_transform(x)
    fitted = perf_counter()
    result = {
        "scores": scores, "pca": pca, "subset": subset, "activation_dir": str(ACTIVATION_DIR),
        "source_row_ids": rows["source_row_id"].to_numpy(copy=True),
        "load_seconds": loaded - started, "pca_seconds": fitted - loaded,
    }
    return layer, result


if min(len(selected) - 1, config["d_model"]) < N_COMPONENTS:
    raise ValueError(f"{N_COMPONENTS} centered PCs require at least {N_COMPONENTS + 1} prompts and {N_COMPONENTS} features")
if not isinstance(N_WORKERS, int) or N_WORKERS < 1:
    raise ValueError("N_WORKERS must be a positive integer")
results = {}
workers = min(N_WORKERS, len(LAYERS))
started = perf_counter()
print(f"Fitting {len(LAYERS)} layers with {workers} parallel workers...", flush=True)
# Separate processes avoid competing BLAS/OpenMP thread pools in the kernel.
with parallel_config(backend="loky", inner_max_num_threads=1):
    completed = Parallel(n_jobs=workers, return_as="generator_unordered")(
        delayed(fit_layer)(layer, selected, SUBSET, SEED) for layer in LAYERS
    )
    for layer, result in completed:
        results[layer] = result
        ratios = result["pca"].explained_variance_ratio_
        print(f"[{len(results):02d}/{len(LAYERS)}] Layer {layer:02d}: "
              f"top {N_COMPONENTS} explain {ratios.sum():.2%}; "
              f"load {result['load_seconds']:.2f}s, PCA {result['pca_seconds']:.2f}s",
              flush=True)
results = dict(sorted(results.items()))
variance = pd.DataFrame({
    layer: result["pca"].explained_variance_ratio_
    for layer, result in results.items()
}, index=[f"PC{i + 1}" for i in range(N_COMPONENTS)]).T
variance.index.name = "layer"
print(f"Finished in {perf_counter() - started:.1f}s", flush=True)

Fitting 32 layers with 8 parallel workers...
[01/32] Layer 00: top 15 explain 84.11%; load 2.24s, PCA 2.26s
[02/32] Layer 04: top 15 explain 88.35%; load 2.05s, PCA 2.19s
[03/32] Layer 05: top 15 explain 91.96%; load 2.08s, PCA 2.14s
[04/32] Layer 06: top 15 explain 93.10%; load 2.26s, PCA 2.18s
[05/32] Layer 02: top 15 explain 85.41%; load 2.74s, PCA 2.21s
[06/32] Layer 07: top 15 explain 93.55%; load 2.65s, PCA 2.15s
[07/32] Layer 03: top 15 explain 88.17%; load 2.84s, PCA 2.42s
[08/32] Layer 01: top 15 explain 79.71%; load 2.65s, PCA 2.47s
[09/32] Layer 10: top 15 explain 93.84%; load 2.22s, PCA 2.19s
[10/32] Layer 09: top 15 explain 93.71%; load 2.39s, PCA 2.31s
[11/32] Layer 08: top 15 explain 94.10%; load 2.75s, PCA 2.21s
[12/32] Layer 12: top 15 explain 94.28%; load 2.13s, PCA 2.27s
[13/32] Layer 13: top 15 explain 94.50%; load 2.14s, PCA 2.22s
[14/32] Layer 11: top 15 explain 93.84%; load 2.37s, PCA 2.24s
[15/32] Layer 14: top 15 explain 95.47%; load 2.25s, PCA 2.24s
[16/32] La

## Interactive 3D projection

Choose `PCS_3D`, for example `(4, 5, 6)` for PCs 4–6, and rerun the cell to plot.
The dropdown selects a layer. Axes and title report the chosen PCs' explained variance.

In [12]:
PCS_3D = (1, 4, 3)  # 1-based PC numbers; choose any 3 distinct PCs from 1–15.
# if len(PCS_3D) != 3 or len(set(PCS_3D)) != 3 or any(
#     not isinstance(pc, int) or not 1 <= pc <= results[LAYERS[0]]["scores"].shape[1]
#     for pc in PCS_3D
# ):
#     raise ValueError("Choose 3 distinct PC numbers within the fitted range")
# pc3 = np.array(PCS_3D) - 1

# for result in results.values():
#     if result["activation_dir"] != str(ACTIVATION_DIR) or result["subset"] != SUBSET or not np.array_equal(
#         result["source_row_ids"], selected["source_row_id"].to_numpy()
#     ):
#         raise ValueError("The selection changed: rerun the PCA fitting cell")

# def token_description(row):
#     return f"token={escape(repr(row.decoded_token))} (index {row.token_index})"


# hover_text = [
#     f"row_id={row.source_row_id}<br>prompt={escape(repr(row.prompt))}"
#     f"<br>{token_description(row)}"
#     f"<br>labels={escape(json.dumps(row.labels, ensure_ascii=False))}"
#     for row in selected.itertuples(index=False)
# ]
# marker = {"size": 3, "opacity": 1.0}
# color_values = labels[color_label]
# numeric = pd.to_numeric(color_values, errors="coerce")
# if color_label == "hex":
#     marker["color"] = color_values.tolist()
# elif numeric.notna().all():
#     marker.update(color=numeric.to_numpy(), colorscale="Viridis", showscale=True,
#                   colorbar={"title": color_label})
# else:
#     categories = sorted(color_values.fillna("(missing)").astype(str).unique())
#     codes = pd.Categorical(color_values.fillna("(missing)").astype(str), categories).codes
#     palette = px.colors.qualitative.Alphabet
#     scale = []
#     for i in range(len(categories)):
#         color = palette[i % len(palette)]
#         scale.extend([(i / len(categories), color), ((i + 1) / len(categories), color)])
#     marker.update(color=codes, colorscale=scale, cmin=-0.5, cmax=len(categories) - 0.5,
#                   showscale=True, colorbar={"title": color_label,
#                   "tickvals": list(range(len(categories))), "ticktext": categories})


# def layer_layout(layer):
#     ratios = results[layer]["pca"].explained_variance_ratio_
#     return {
#         "title": {"text": f"{SUBSET} · last token · layer {layer:02d} · PCs {PCS_3D} explain {ratios[pc3].sum():.1%}"},
#         "scene": {"xaxis_title": f"PC{PCS_3D[0]} ({ratios[pc3[0]]:.1%})",
#                   "yaxis_title": f"PC{PCS_3D[1]} ({ratios[pc3[1]]:.1%})",
#                   "zaxis_title": f"PC{PCS_3D[2]} ({ratios[pc3[2]]:.1%})", "aspectmode": "data"},
#     }

# first = results[LAYERS[0]]["scores"][:, pc3]
# fig = go.Figure(go.Scatter3d(
#     x=first[:, 0], y=first[:, 1], z=first[:, 2], mode="markers", marker=marker,
#     text=hover_text,
#     hovertemplate=("%{text}"
#                    f"<br>PC{PCS_3D[0]}=%{{x:.3f}}"
#                    f"<br>PC{PCS_3D[1]}=%{{y:.3f}}"
#                    f"<br>PC{PCS_3D[2]}=%{{z:.3f}}"
#                    "<extra></extra>"),
# ))
# buttons = []
# for layer in LAYERS:
#     scores = results[layer]["scores"][:, pc3]
#     buttons.append({"label": f"Layer {layer:02d}", "method": "update", "args": [
#         {"x": [scores[:, 0]], "y": [scores[:, 1]], "z": [scores[:, 2]]},
#         layer_layout(layer),
#     ]})
# fig.update_layout(**layer_layout(LAYERS[0]), height=720,
#                   margin={"l": 0, "r": 0, "b": 0, "t": 100},
#                   updatemenus=[{"buttons": buttons, "x": 0, "y": 1.10,
#                                 "xanchor": "left", "yanchor": "top"}])
# fig.show()

## Interactive 2D projection

Choose `PCS_2D`, for example `(4, 5)` for PCs 4–5, and rerun the cell to plot.
Run the 3D cell once first to prepare shared colors and hover text. Neither plot refits PCA.

In [13]:
# PCS_2D = (1, 2)  # 1-based PC numbers; choose any 2 distinct PCs from 1–15.
# if len(PCS_2D) != 2 or len(set(PCS_2D)) != 2 or any(
#     not isinstance(pc, int) or not 1 <= pc <= results[LAYERS[0]]["scores"].shape[1]
#     for pc in PCS_2D
# ):
#     raise ValueError("Choose 2 distinct PC numbers within the fitted range")
# pc2 = np.array(PCS_2D) - 1

# for result in results.values():
#     if result["activation_dir"] != str(ACTIVATION_DIR) or result["subset"] != SUBSET or not np.array_equal(
#         result["source_row_ids"], selected["source_row_id"].to_numpy()
#     ):
#         raise ValueError("The selection changed: rerun the PCA fitting and 3D plotting cells")


# def layer_layout_2d(layer):
#     ratios = results[layer]["pca"].explained_variance_ratio_
#     return {
#         "title": {"text": f"{SUBSET} · last token · layer {layer:02d} · PCs {PCS_2D} explain {ratios[pc2].sum():.1%}"},
#         "xaxis": {"title": f"PC{PCS_2D[0]} ({ratios[pc2[0]]:.1%})", "autorange": True},
#         "yaxis": {"title": f"PC{PCS_2D[1]} ({ratios[pc2[1]]:.1%})", "autorange": True,
#                   "scaleanchor": "x", "scaleratio": 1},
#     }


# first_2d = results[LAYERS[0]]["scores"][:, pc2]
# fig_2d = go.Figure(go.Scattergl(
#     x=first_2d[:, 0], y=first_2d[:, 1], mode="markers",
#     marker={**marker, "size": 6}, text=hover_text,
#     hovertemplate=("%{text}"
#                    f"<br>PC{PCS_2D[0]}=%{{x:.3f}}"
#                    f"<br>PC{PCS_2D[1]}=%{{y:.3f}}"
#                    "<extra></extra>"),
# ))
# buttons_2d = []
# for layer in LAYERS:
#     scores = results[layer]["scores"][:, pc2]
#     buttons_2d.append({"label": f"Layer {layer:02d}", "method": "update", "args": [
#         {"x": [scores[:, 0]], "y": [scores[:, 1]]},
#         layer_layout_2d(layer),
#     ]})
# fig_2d.update_layout(**layer_layout_2d(LAYERS[0]), height=720,
#                      margin={"l": 60, "r": 30, "b": 60, "t": 100},
#                      updatemenus=[{"buttons": buttons_2d, "x": 0, "y": 1.10,
#                                    "xanchor": "left", "yanchor": "top"}])
# fig_2d.show()


## K3D explorer

Run after the PCA fit; the Plotly cells are optional. Select any three PCs and layer
using the controls. The size slider sets point diameter as a fraction of the largest
axis span. Coordinates keep their original scale; features are not standardized.
Points are fully opaque, with the same numeric `COLOR_LABEL` and color range across layers.

This uses the project's `k3d` and `ipywidgets` dependencies in the notebook kernel.
[Point rendering reference](https://k3d-jupyter.org/reference/factory.points.html).

In [14]:
import k3d
import ipywidgets as widgets

for result in results.values():
    if result["activation_dir"] != str(ACTIVATION_DIR) or result["subset"] != SUBSET or not np.array_equal(
        result["source_row_ids"], selected["source_row_id"].to_numpy()
    ):
        raise ValueError("The selection changed: rerun the PCA fitting cell")

k3d_colors = pd.to_numeric(labels[color_label], errors="raise").to_numpy(dtype=np.float32)
if not np.isfinite(k3d_colors).all():
    raise ValueError("K3D coloring requires a finite numeric COLOR_LABEL")

layer_control = widgets.Dropdown(options=LAYERS, value=LAYERS[0], description="Layer")
pc_options = list(range(1, results[LAYERS[0]]["scores"].shape[1] + 1))
pc_controls = [widgets.Dropdown(options=pc_options, value=i + 1, description=axis)
               for i, axis in enumerate(["X PC", "Y PC", "Z PC"])]
size_control = widgets.FloatLogSlider(value=0.003, base=10, min=-4, max=-1,
                                     step=0.1, description="Point size", continuous_update=False)
status = widgets.HTML()

initial = np.asarray(results[LAYERS[0]]["scores"][:, :3], dtype=np.float32)
k3d_cloud = k3d.points(
    initial, attribute=k3d_colors,
    color_map=k3d.colormaps.matplotlib_color_maps.Viridis,
    color_range=[float(k3d_colors.min()), float(k3d_colors.max())],
    shader="3d", opacity=1.0, name=color_label,
    point_size=max(float(np.ptp(initial, axis=0).max()) * size_control.value, 1e-6),
)
k3d_plot = k3d.plot(height=720, camera_auto_fit=False)
k3d_plot += k3d_cloud
k3d_plot.colorbar_object_id = k3d_cloud.id


def update_k3d(change=None):
    pcs = [control.value for control in pc_controls]
    if len(set(pcs)) != 3:
        status.value = "Choose three distinct PCs; the previous projection is still displayed."
        return
    layer = layer_control.value
    indices = np.array(pcs) - 1
    scores = np.asarray(results[layer]["scores"][:, indices], dtype=np.float32)
    ratios = results[layer]["pca"].explained_variance_ratio_[indices]
    k3d_cloud.positions = scores
    k3d_cloud.point_size = max(float(np.ptp(scores, axis=0).max()) * size_control.value, 1e-6)
    k3d_plot.axes = [f"PC{pc} ({ratio:.1%})" for pc, ratio in zip(pcs, ratios)]
    status.value = f"Layer {layer:02d} · PCs {pcs} · {ratios.sum():.1%} explained · {len(scores):,} prompts"
    if change is None or change["owner"] is not size_control:
        k3d_plot.camera_reset()


for control in [layer_control, *pc_controls, size_control]:
    control.observe(update_k3d, names="value")
update_k3d()
display(widgets.VBox([widgets.HBox([layer_control, *pc_controls]), size_control, status]))
k3d_plot.display()

Output()